# 9. Joining library data to ward boundaries

A common urban-data workflow is:

```text
table with useful statistics
        +
geographic boundary file
        ↓
one GeoDataFrame containing both
```

Here:

- the **library CSV** contains one row per branch;
- the **ward GeoPackage** contains one row per ward polygon.

We want to summarize the libraries by ward and attach those ward-level statistics to the polygons.

## First intuition: the datasets are currently at different levels

### Library table

| Branch | WardNo | Workstations |
|---|---:|---:|
| Branch A | 1 | 14 |
| Branch B | 1 | 20 |
| Branch C | 2 | 9 |

There can be **many rows per ward**.

### Ward boundary file

| WARD_SHORT_CODE | WARD_NAME | geometry |
|---|---|---|
| 01 | Ward 1 | POLYGON(...) |
| 02 | Ward 2 | POLYGON(...) |

There is **one row per ward**.

Before joining, both datasets should represent the same unit: one row per ward.

In [ ]:
import pandas as pd
import geopandas as gpd

libraries = pd.read_csv(
    "../data/tpl-branch-general-information-2023.csv"
)

wards = gpd.read_file(
    "../data/city-wards.gpkg"
)

## Step 1: inspect possible join fields

The library data contains `WardNo` and `WardName`.

The boundary data contains `WARD_SHORT_CODE` and `WARD_NAME`.

Similar names do not guarantee compatible values, so inspect both the values and data types.

In [ ]:
display(
    libraries[
        ["BranchName", "WardNo", "WardName", "Workstations"]
    ].head()
)

display(
    wards[
        ["WARD_SHORT_CODE", "WARD_NAME"]
    ].head()
)

print("WardNo dtype:", libraries["WardNo"].dtype)
print("WARD_SHORT_CODE dtype:", wards["WARD_SHORT_CODE"].dtype)

## Step 2: aggregate the branch-level table

We want:

| Branch | Ward | Workstations |
|---|---:|---:|
| A | 1 | 14 |
| B | 1 | 20 |
| C | 2 | 9 |

to become:

| Ward | library_count | total_workstations |
|---|---:|---:|
| 1 | 2 | 34 |
| 2 | 1 | 9 |

`groupby()` identifies each ward, and `.agg()` tells pandas what summary to calculate within each group.

In [ ]:
physical_libraries = libraries.loc[
    libraries["PhysicalBranch"] == 1
].copy()

ward_library_summary = (
    physical_libraries
    .groupby("WardNo")
    .agg(
        library_count=("BranchName", "count"),
        total_workstations=("Workstations", "sum")
    )
    .reset_index()
)

ward_library_summary.head()

## Step 3: make the identifiers compatible

The datasets refer to the same wards but format the IDs differently.

| Library summary | Ward boundaries |
|---|---|
| `1.0` | `"01"` |
| `2.0` | `"02"` |
| `10.0` | `"10"` |

A join compares values directly. Python does not infer that `1.0` and `"01"` mean the same thing.

We therefore create a matching text identifier.

In [ ]:
ward_library_summary["WARD_SHORT_CODE"] = (
    ward_library_summary["WardNo"]
    .astype(int)
    .astype(str)
    .str.zfill(2)
)

ward_library_summary[
    ["WardNo", "WARD_SHORT_CODE"]
].head()

## Step 4: choose the join direction

We want to keep **every ward polygon**, so the ward GeoDataFrame goes on the left.

Before:

### Wards

| WARD_SHORT_CODE | geometry |
|---|---|
| 01 | POLYGON(...) |
| 02 | POLYGON(...) |
| 03 | POLYGON(...) |

### Library summary

| WARD_SHORT_CODE | library_count |
|---|---:|
| 01 | 3 |
| 03 | 5 |

After a **left join**:

| WARD_SHORT_CODE | library_count | geometry |
|---|---:|---|
| 01 | 3 | POLYGON(...) |
| 02 | missing | POLYGON(...) |
| 03 | 5 | POLYGON(...) |

The geometry remains because `wards` is the left-hand GeoDataFrame.

In [ ]:
wards_with_libraries = wards.merge(
    ward_library_summary,
    on="WARD_SHORT_CODE",
    how="left"
)

wards_with_libraries[
    [
        "WARD_SHORT_CODE",
        "WARD_NAME",
        "library_count",
        "total_workstations",
        "geometry"
    ]
].head()

## Step 5: check the join

A merge can run without an error and still be wrong.

| Check | Question |
|---|---|
| Row count | Did we unexpectedly gain or lose wards? |
| Missing values | Did some IDs fail to match? |
| Duplicate IDs | Did one-to-many matching duplicate rows? |
| Spot checks | Do familiar wards have plausible values? |

Because the boundary file contains 25 wards, the joined result should still contain 25 rows.

In [ ]:
print("Rows before join:", len(wards))
print("Rows after join:", len(wards_with_libraries))

print("\nMissing joined values:")
print(
    wards_with_libraries[
        ["library_count", "total_workstations"]
    ].isna().sum()
)

## Attribute join versus spatial join

This is an **attribute join**:

```text
ward ID  ↔  ward ID
```

We matched records using a shared identifier.

A **spatial join** would instead ask something like:

```text
Which ward polygon contains this library point?
```

Attribute joins are preferable when a reliable shared identifier already exists because they are simpler and more explicit.